# 14 — v14: New Feature Engineering on v9 Base

**Current best:** v9 (3-model Ridge stack XGB+LGBM+ET) → Kaggle RMSE **21,755** | Goal: break **21,500**.

## What changes vs v9

| # | Change | Why |
|---|---|---|
| 1 | **Remaining lease non-linearity** | CPF cannot be used below 60-year lease → real price cliff. `log_remaining_lease`, `lease_below_60` flag, `lease_x_below60` interaction. |
| 2 | **Storey × flat_type interaction** | High floors command bigger premiums for large flats (5-room/exec) than small ones. `storey_x_flattype = mid_storey × flat_type_rank`. |
| 3 | **Time-varying area price (OOF)** | `town_median_price` is a global median — doesn't capture appreciation trends. OOF (town, year) encoding gives a time-specific price signal per area. |
| 4 | **Block-level OOF encoding** | `block_num` is just an extracted integer. `(town, block)` OOF median captures block-specific pricing within each town. |
| 5 | **OOF loop uses train_fe** | All ENCODE_PAIRS now use `fe_tr = train_fe.iloc[tr_idx]` — works even if group cols are in DROP_ALL, prevents index misalignment. |

---
## 1. Imports & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from math import radians, cos, sin, asin, sqrt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

train = pd.read_csv('../../data/raw/train.csv', low_memory=False)
test  = pd.read_csv('../../data/raw/test.csv',  low_memory=False)
print(f'Train: {train.shape}  |  Test: {test.shape}')

Train: (150634, 77)  |  Test: (16737, 76)


---
## 2. Feature Engineering

Same as v8, plus three new features:
- **`postal_sector`** — first 4 chars of `postal` (598 unique sectors, 100% test coverage)
- **`block_num`** — numeric part of `block` string (e.g. "123A" → 123)
- `postal` and `block` (raw) will be dropped in Section 4

In [2]:
DROP_COLS   = ['floor_area_sqft','lower','upper','mid','full_flat_type',
               'address','Tranc_YearMonth','residential','year_completed']
SOLD_COLS   = ['1room_sold','2room_sold','3room_sold','4room_sold',
               '5room_sold','exec_sold','multigen_sold','studio_apartment_sold']
RENTAL_COLS = ['1room_rental','2room_rental','3room_rental','other_room_rental']
FILL_ZERO   = ['Hawker_Within_500m','Mall_Within_500m','Hawker_Within_1km',
               'Mall_Within_1km','Hawker_Within_2km','Mall_Within_2km']
MATURE_ESTATES = {
    'ANG MO KIO','BEDOK','BISHAN','BUKIT MERAH','BUKIT TIMAH',
    'CENTRAL AREA','CLEMENTI','GEYLANG','KALLANG/WHAMPOA',
    'MARINE PARADE','PASIR RIS','QUEENSTOWN','SERANGOON','TAMPINES','TOA PAYOH'
}
ROOM_COUNT = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
              '5 ROOM':5,'EXECUTIVE':5,'MULTI-GENERATION':6}
FLAT_TYPE_RANK = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
                  '5 ROOM':5,'EXECUTIVE':6,'MULTI-GENERATION':7}
CBD_LAT, CBD_LON = 1.2847, 103.8510

STREET_FREQ = train['street_name'].value_counts().to_dict()

def haversine_km(lat, lon, lat2=CBD_LAT, lon2=CBD_LON):
    R = 6371
    lat, lon, lat2, lon2 = map(radians, [lat, lon, lat2, lon2])
    a = sin((lat2-lat)/2)**2 + cos(lat)*cos(lat2)*sin((lon2-lon)/2)**2
    return 2*R*asin(sqrt(a))

def engineer_features(df, amenity_caps=None):
    df = df.copy()
    for c in FILL_ZERO:
        df[c] = df[c].fillna(0)

    # Existing features (same as v9)
    df['remaining_lease']  = 99 - (df['Tranc_Year'] - df['lease_commence_date'])
    df['dist_to_cbd']      = df.apply(lambda r: haversine_km(r['Latitude'], r['Longitude']), axis=1)
    df['is_mature_estate'] = df['town'].str.upper().isin(MATURE_ESTATES).astype(int)
    df['tranc_month_sin']     = np.sin(2*np.pi*df['Tranc_Month']/12)
    df['tranc_month_cos']     = np.cos(2*np.pi*df['Tranc_Month']/12)
    df['total_sold']          = df[SOLD_COLS].sum(axis=1)
    df['total_rental']        = df[RENTAL_COLS].sum(axis=1)
    df['rental_ratio']        = (df['total_rental'] / df['total_dwelling_units'].replace(0, np.nan)).fillna(0)
    df['num_rooms']           = df['flat_type'].str.upper().map(ROOM_COUNT).fillna(4)
    df['floor_area_per_room'] = df['floor_area_sqm'] / df['num_rooms']
    caps = amenity_caps or {}
    new_caps = {}
    for col in ['mrt_nearest_distance','Mall_Nearest_Distance','Hawker_Nearest_Distance']:
        cap = caps.get(col, df[col].quantile(0.99))
        new_caps[col] = cap
        inv = 1 / df[col].clip(1, cap)
        inv_min, inv_max = inv.min(), inv.max()
        df[f'_inv_{col}'] = (inv - inv_min) / (inv_max - inv_min + 1e-9)
    df['amenity_score'] = (df['_inv_mrt_nearest_distance'] +
                           df['_inv_Mall_Nearest_Distance'] +
                           df['_inv_Hawker_Nearest_Distance']) / 3
    df.drop(columns=[c for c in df.columns if c.startswith('_inv_')], inplace=True)
    df['dist_x_storey']   = df['dist_to_cbd'] * df['mid_storey']
    df['lease_x_area']    = df['remaining_lease'] * df['floor_area_sqm']
    df['log_dist_to_cbd'] = np.log1p(df['dist_to_cbd'])
    df['street_freq']     = df['street_name'].map(STREET_FREQ).fillna(0)

    # From v9: postal_sector and block_num
    df['postal_sector'] = df['postal'].astype(str).str[:4]
    df['block_num']     = pd.to_numeric(
        df['block'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
    ).fillna(0)

    # NEW v14 — Remaining lease non-linearity (CPF threshold at 60 years)
    df['log_remaining_lease'] = np.log1p(df['remaining_lease'])
    df['lease_below_60']      = (df['remaining_lease'] < 60).astype(int)
    df['lease_x_below60']     = df['remaining_lease'] * df['lease_below_60']

    # NEW v14 — Storey x flat_type interaction
    df['flat_type_rank']    = df['flat_type'].str.upper().map(FLAT_TYPE_RANK).fillna(3)
    df['storey_x_flattype'] = df['mid_storey'] * df['flat_type_rank']

    # NEW v14 — Group keys for time-varying + block-level OOF encoding (dropped from X)
    df['town_year']  = df['town'].astype(str) + '_' + df['Tranc_Year'].astype(str)
    df['town_block'] = df['town'].astype(str) + '_' + df['block_num'].astype(str)

    return df, new_caps

train_fe, train_caps = engineer_features(train)
test_fe,  _          = engineer_features(test, amenity_caps=train_caps)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')
print(f'New v14 features: log_remaining_lease, lease_below_60, lease_x_below60, flat_type_rank, storey_x_flattype')
print(f'Group keys: town_year ({train_fe["town_year"].nunique()} unique), town_block ({train_fe["town_block"].nunique()} unique)')

Train: (150634, 101)  |  Test: (16737, 100)
New v14 features: log_remaining_lease, lease_below_60, lease_x_below60, flat_type_rank, storey_x_flattype
Group keys: town_year (260 unique), town_block (7074 unique)


---
## 3. Per-Fold OOF Target Encoding

Generalised helper replaces `oof_town_median()` from v8. Apply to:
- `town` → `town_median_price` (as before)
- `postal_sector` → `postal_sector_median_price` (**new**, biggest expected gain)
- `flat_model` → `flat_model_median_price` (**new**)

> **No leakage**: each val row's encoded value is computed from the other 4 folds only.

In [3]:
def oof_group_median(group_series, y_series, n_splits=5, random_state=42):
    """OOF median target encoding for any group column."""
    groups = group_series.values
    y      = y_series.values
    encoded   = np.zeros(len(groups))
    global_med = np.median(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in kf.split(groups):
        fold_map = {}
        for g, p in zip(groups[tr_idx], y[tr_idx]):
            fold_map.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_map.items()}
        for i in va_idx:
            encoded[i] = fold_map.get(groups[i], global_med)
    return encoded

# --- Training set: OOF encoding (no leakage) ---
train_fe['town_median_price']          = oof_group_median(train_fe['town'],          train['resale_price'])
train_fe['postal_sector_median_price'] = oof_group_median(train_fe['postal_sector'], train['resale_price'])
train_fe['flat_model_median_price']    = oof_group_median(train_fe['flat_model'],    train['resale_price'])
train_fe['town_year_median_price']     = oof_group_median(train_fe['town_year'],     train['resale_price'])  # NEW v14
train_fe['town_block_median_price']    = oof_group_median(train_fe['town_block'],    train['resale_price'])  # NEW v14

# --- Test set: full-training-set maps (no leakage; test labels unknown) ---
_tmp = pd.DataFrame({
    'town':          train_fe['town'].values,
    'postal_sector': train_fe['postal_sector'].values,
    'flat_model':    train_fe['flat_model'].values,
    'town_year':     train_fe['town_year'].values,
    'town_block':    train_fe['town_block'].values,
    'resale_price':  train['resale_price'].values,
})
global_price_med = train['resale_price'].median()

full_town_map   = _tmp.groupby('town')['resale_price'].median()
full_sector_map = _tmp.groupby('postal_sector')['resale_price'].median()
full_model_map  = _tmp.groupby('flat_model')['resale_price'].median()
full_ty_map     = _tmp.groupby('town_year')['resale_price'].median()   # NEW v14
full_tb_map     = _tmp.groupby('town_block')['resale_price'].median()  # NEW v14

test_fe['town_median_price']          = test_fe['town'].map(full_town_map).fillna(global_price_med)
test_fe['postal_sector_median_price'] = test_fe['postal_sector'].map(full_sector_map).fillna(global_price_med)
test_fe['flat_model_median_price']    = test_fe['flat_model'].map(full_model_map).fillna(global_price_med)
test_fe['town_year_median_price']     = test_fe['town_year'].map(full_ty_map).fillna(global_price_med)    # NEW v14
test_fe['town_block_median_price']    = test_fe['town_block'].map(full_tb_map).fillna(global_price_med)   # NEW v14

print('OOF encoding done: 5 encoded columns')
print(f'town_year_median_price  nulls — train: {train_fe["town_year_median_price"].isna().sum()}, test: {test_fe["town_year_median_price"].isna().sum()}')
print(f'town_block_median_price nulls — train: {train_fe["town_block_median_price"].isna().sum()}, test: {test_fe["town_block_median_price"].isna().sum()}')

OOF encoding done: 5 encoded columns
town_year_median_price  nulls — train: 0, test: 0
town_block_median_price nulls — train: 0, test: 0


---
## 4. Prepare X and y

`postal` and `block` (raw, high-cardinality) are dropped — replaced by `postal_sector_median_price` and `block_num`.

In [4]:
DROP_ALL = (['id', 'resale_price', 'postal', 'block', 'town_year', 'town_block']
            + DROP_COLS + SOLD_COLS + RENTAL_COLS + ['num_rooms'])

X      = train_fe.drop(columns=DROP_ALL, errors='ignore')
y      = train['resale_price'].values
y_log  = np.log1p(y)

X_test = test_fe.drop(columns=[c for c in DROP_ALL if c != 'resale_price'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f'Features: {X.shape[1]}  (num={len(num_cols)}, cat={len(cat_cols)})')
v14_new = [c for c in ['log_remaining_lease','lease_below_60','lease_x_below60',
                        'flat_type_rank','storey_x_flattype',
                        'town_year_median_price','town_block_median_price'] if c in num_cols]
print(f'New v14 numeric features in X: {v14_new}')
print(f'town_year/town_block correctly dropped: {("town_year" not in X.columns) and ("town_block" not in X.columns)}')
assert X_test.shape[1] == X.shape[1], 'X/X_test column mismatch'

Features: 78  (num=63, cat=15)
New v14 numeric features in X: ['log_remaining_lease', 'lease_below_60', 'lease_x_below60', 'flat_type_rank', 'storey_x_flattype', 'town_year_median_price', 'town_block_median_price']
town_year/town_block correctly dropped: True


---
## 5. Preprocessing Pipeline

In [5]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def neg_dollar_rmse(y_log_true, y_log_pred):
    return -np.sqrt(mean_squared_error(np.expm1(y_log_true), np.expm1(y_log_pred)))

dollar_rmse_scorer = make_scorer(neg_dollar_rmse)
print('Preprocessor ready.')

Preprocessor ready.


---
## 6. RandomizedSearchCV — XGBoost

Wider search space than v8 (40 trials vs 30, more parameter options) using the same `dollar_rmse_scorer` (RMSE in SGD, not log-space).

**Key additions vs v8 grid:**
- `n_estimators` extended to 2000
- `reg_alpha` and `reg_lambda` now explicitly searched (v8 had fixed defaults)
- `colsample_bytree` includes 0.4–0.5 range (sometimes helps with many features)

In [6]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

param_dist_xgb = {
    'model__n_estimators'     : [500, 800, 1000, 1500, 2000],
    'model__max_depth'        : [4, 5, 6, 7, 8, 9],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.07, 0.1],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.4, 0.5, 0.6, 0.7, 0.8],
    'model__min_child_weight' : [1, 3, 5, 7, 10],
    'model__reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'model__reg_lambda'       : [0.5, 1.0, 1.5, 2.0, 3.0],
}
xgb_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor),
              ('model', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, tree_method='hist'))]),
    param_distributions=param_dist_xgb, n_iter=40, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
xgb_search.fit(X_train, y_train_log)
print(f'\nXGB best CV RMSE: ${-xgb_search.best_score_:,.0f}')
print('Best params:', xgb_search.best_params_)

XGB_PARAMS = {k.replace('model__',''):v for k,v in xgb_search.best_params_.items()}
XGB_PARAMS.update({'random_state':42, 'n_jobs':-1, 'verbosity':0, 'tree_method':'hist'})

xgb_check = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
xgb_check.fit(X_train, y_train_log)
xgb_val_rmse = rmse(y_val, np.expm1(xgb_check.predict(X_val)))
print(f'XGB val RMSE: ${xgb_val_rmse:,.0f}  (v8 was ~$21,800–22,500)')

Fitting 3 folds for each of 40 candidates, totalling 120 fits

XGB best CV RMSE: $22,854
Best params: {'model__subsample': 0.9, 'model__reg_lambda': 1.5, 'model__reg_alpha': 0.01, 'model__n_estimators': 2000, 'model__min_child_weight': 7, 'model__max_depth': 7, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.4}
XGB val RMSE: $21,911  (v8 was ~$21,800–22,500)


---
## 7. RandomizedSearchCV — LightGBM

Extended grid with `num_leaves` up to 300 and explicit regularisation search.

In [7]:
param_dist_lgbm = {
    'model__n_estimators'     : [500, 800, 1000, 1500, 2000],
    'model__max_depth'        : [5, 6, 7, 8, 10, 12],
    'model__num_leaves'       : [60, 80, 100, 127, 200, 300],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.07, 0.1],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.5, 0.6, 0.7, 0.8],
    'model__min_child_samples': [10, 20, 30, 50],
    'model__reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'model__reg_lambda'       : [0, 0.1, 0.5, 1.0],
}
lgbm_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor),
              ('model', LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1))]),
    param_distributions=param_dist_lgbm, n_iter=40, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
lgbm_search.fit(X_train, y_train_log)
print(f'\nLGBM best CV RMSE: ${-lgbm_search.best_score_:,.0f}')
print('Best params:', lgbm_search.best_params_)

LGBM_PARAMS = {k.replace('model__',''):v for k,v in lgbm_search.best_params_.items()}
LGBM_PARAMS.update({'random_state':42, 'n_jobs':-1, 'verbosity':-1})

lgbm_check = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
lgbm_check.fit(X_train, y_train_log)
lgbm_val_rmse = rmse(y_val, np.expm1(lgbm_check.predict(X_val)))
print(f'LGBM val RMSE: ${lgbm_val_rmse:,.0f}')

Fitting 3 folds for each of 40 candidates, totalling 120 fits

LGBM best CV RMSE: $22,863
Best params: {'model__subsample': 0.7, 'model__reg_lambda': 0.5, 'model__reg_alpha': 1.0, 'model__num_leaves': 127, 'model__n_estimators': 1000, 'model__min_child_samples': 50, 'model__max_depth': 12, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.5}
LGBM val RMSE: $22,031


---
## 8. RandomizedSearchCV — Extra Trees

ET has no learning rate interaction so Bayesian search adds little. Keep RandomizedSearchCV.

In [9]:
param_dist_et = {                                                                      
      'model__n_estimators'     : [200, 300, 500],                                                       
      'model__max_depth'        : [15, 20, 30],      # remove None — it's the slow killer
      'model__min_samples_split': [2, 5, 10],                                                            
      'model__min_samples_leaf' : [1, 2, 4],                                                             
      'model__max_features'     : [0.5, 0.6, 0.7, 0.8],                                                  
  }                                                                                                      
et_search = RandomizedSearchCV(                                                        
      Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(random_state=42, n_jobs=-1))]),     
      param_distributions=param_dist_et, n_iter=15, cv=3,   # 15 trials instead of 25                    
      scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,                     
  )
et_search.fit(X_train, y_train_log)
print(f'ET best CV RMSE: ${-et_search.best_score_:,.0f}')
et_val_rmse = rmse(y_val, np.expm1(et_search.best_estimator_.predict(X_val)))
print(f'ET val RMSE:     ${et_val_rmse:,.0f}')

ET_PARAMS = {k.replace('model__',''):v for k,v in et_search.best_params_.items()}
ET_PARAMS.update({'random_state':42,'n_jobs':-1})
print(f'ET_PARAMS: {ET_PARAMS}')

Fitting 3 folds for each of 15 candidates, totalling 45 fits
ET best CV RMSE: $24,589
ET val RMSE:     $23,338
ET_PARAMS: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 30, 'random_state': 42, 'n_jobs': -1}


---
## 9. Generate OOF Predictions (5-Fold) — 3 Models

Per-fold target encoding is **recomputed inside each fold** from fold-train rows only.
This now covers `town`, `postal_sector`, and `flat_model`.

In [8]:
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof  = np.zeros(len(X))
lgbm_oof = np.zeros(len(X))
et_oof   = np.zeros(len(X))

xgb_test_folds  = np.zeros((5, len(X_test)))
lgbm_test_folds = np.zeros((5, len(X_test)))
et_test_folds   = np.zeros((5, len(X_test)))

fold_rmses_xgb  = []
fold_rmses_lgbm = []
fold_rmses_et   = []

# (group_col, price_col, min_count) — group_col accessed via train_fe, not X
ENCODE_PAIRS = [
    ('town',          'town_median_price',          1),
    ('postal_sector', 'postal_sector_median_price', 1),
    ('flat_model',    'flat_model_median_price',    1),
    ('town_year',     'town_year_median_price',     5),  # NEW v14
    ('town_block',    'town_block_median_price',    3),  # NEW v14
]

print('Generating OOF predictions (5-fold, 3 models)...')
print(f'{"Fold":>5}  {"XGB RMSE":>12}  {"LGBM RMSE":>12}  {"ET RMSE":>12}')
print('-' * 55)

for fold, (tr_idx, va_idx) in enumerate(kf5.split(X)):
    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr, y_va = y[tr_idx], y[va_idx]
    y_tr_log   = np.log1p(y_tr)
    global_med_tr = np.median(y_tr)

    fe_tr = train_fe.iloc[tr_idx]  # always use train_fe for group col lookups
    fe_va = train_fe.iloc[va_idx]

    # Recompute per-fold target encodings (prevents leakage)
    for group_col, price_col, min_ct in ENCODE_PAIRS:
        fold_map = {}
        for g, p in zip(fe_tr[group_col].values, y_tr):
            fold_map.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_map.items() if len(ps) >= min_ct}
        X_tr[price_col] = fe_tr[group_col].map(fold_map).fillna(global_med_tr).values
        X_va[price_col] = fe_va[group_col].map(fold_map).fillna(global_med_tr).values

    # XGBoost
    xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
    xgb_pipe.fit(X_tr, y_tr_log)
    xgb_oof[va_idx]      = xgb_pipe.predict(X_va)
    xgb_test_folds[fold] = xgb_pipe.predict(X_test)
    fold_rmses_xgb.append(rmse(y_va, np.expm1(xgb_oof[va_idx])))

    # LightGBM
    lgbm_pipe = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
    lgbm_pipe.fit(X_tr, y_tr_log)
    lgbm_oof[va_idx]      = lgbm_pipe.predict(X_va)
    lgbm_test_folds[fold] = lgbm_pipe.predict(X_test)
    fold_rmses_lgbm.append(rmse(y_va, np.expm1(lgbm_oof[va_idx])))

    # Extra Trees
    et_pipe = Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(**ET_PARAMS))])
    et_pipe.fit(X_tr, y_tr_log)
    et_oof[va_idx]      = et_pipe.predict(X_va)
    et_test_folds[fold] = et_pipe.predict(X_test)
    fold_rmses_et.append(rmse(y_va, np.expm1(et_oof[va_idx])))

    print(f'{fold+1:>5}  ${fold_rmses_xgb[-1]:>10,.0f}  ${fold_rmses_lgbm[-1]:>10,.0f}  ${fold_rmses_et[-1]:>10,.0f}')

print('-' * 55)
print(f'{"Mean":>5}  ${np.mean(fold_rmses_xgb):>10,.0f}  ${np.mean(fold_rmses_lgbm):>10,.0f}  ${np.mean(fold_rmses_et):>10,.0f}')

xgb_test_avg  = xgb_test_folds.mean(axis=0)
lgbm_test_avg = lgbm_test_folds.mean(axis=0)
et_test_avg   = et_test_folds.mean(axis=0)

print('\nOOF generation complete.')

Generating OOF predictions (5-fold, 3 models)...
 Fold      XGB RMSE     LGBM RMSE       ET RMSE
-------------------------------------------------------


NameError: name 'ET_PARAMS' is not defined

---
## 10. Sanity Check — Individual Models & Blends

In [11]:
print('Individual OOF RMSE:')
print(f'  XGB:  ${rmse(y, np.expm1(xgb_oof)):>8,.0f}')
print(f'  LGBM: ${rmse(y, np.expm1(lgbm_oof)):>8,.0f}')
print(f'  ET:   ${rmse(y, np.expm1(et_oof)):>8,.0f}')

blend_equal = np.expm1((xgb_oof + lgbm_oof + et_oof) / 3)
print(f'\n3-model equal-weight blend OOF RMSE: ${rmse(y, blend_equal):>8,.0f}')
print(f'v8 OOF RMSE (for comparison):         ~$21,804')

Individual OOF RMSE:
  XGB:  $  21,904
  LGBM: $  21,803
  ET:   $  23,589

3-model equal-weight blend OOF RMSE: $  21,757
v8 OOF RMSE (for comparison):         ~$21,804


---
## 11. Ridge Meta-Model on 3 OOF Features

Ridge learns optimal weights for the 3 base model predictions.
Check that all 3 coefficients are positive — confirms each model contributes genuinely.

In [12]:
meta_X_train = np.column_stack([xgb_oof, lgbm_oof, et_oof])
meta_X_test  = np.column_stack([xgb_test_avg, lgbm_test_avg, et_test_avg])

print('Ridge alpha sweep (3 meta-features):')
print(f'{"alpha":>10}  {"RMSE":>12}  {"XGB coef":>10}  {"LGBM coef":>10}  {"ET coef":>10}')
print('-' * 62)

best_meta_rmse   = float('inf')
best_alpha_ridge = 1.0

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(meta_X_train, y_log)
    meta_pred_log = ridge.predict(meta_X_train)
    meta_rmse_val = rmse(y, np.expm1(meta_pred_log))
    print(f'{alpha:>10.3f}  ${meta_rmse_val:>10,.0f}  {ridge.coef_[0]:>10.4f}  {ridge.coef_[1]:>10.4f}  {ridge.coef_[2]:>10.4f}')
    if meta_rmse_val < best_meta_rmse:
        best_meta_rmse   = meta_rmse_val
        best_alpha_ridge = alpha

print(f'\nBest Ridge alpha: {best_alpha_ridge}')
meta_model = Ridge(alpha=best_alpha_ridge)
meta_model.fit(meta_X_train, y_log)
print(f'Learned coefficients:')
print(f'  XGB:  {meta_model.coef_[0]:.4f}')
print(f'  LGBM: {meta_model.coef_[1]:.4f}')
print(f'  ET:   {meta_model.coef_[2]:.4f}')
print(f'  Intercept: {meta_model.intercept_:.4f}')
print(f'\nOOF meta-train RMSE (optimistic): ${best_meta_rmse:,.0f}')
print(f'v8 Kaggle RMSE for comparison:      $21,804.67')

Ridge alpha sweep (3 meta-features):
     alpha          RMSE    XGB coef   LGBM coef     ET coef
--------------------------------------------------------------
     0.001  $    21,627      0.3854      0.4824      0.1347
     0.010  $    21,627      0.3854      0.4823      0.1348
     0.100  $    21,627      0.3858      0.4815      0.1352
     1.000  $    21,628      0.3891      0.4744      0.1390
    10.000  $    21,632      0.3983      0.4346      0.1696
   100.000  $    21,686      0.3666      0.3673      0.2670

Best Ridge alpha: 0.001
Learned coefficients:
  XGB:  0.3854
  LGBM: 0.4824
  ET:   0.1347
  Intercept: -0.0318

OOF meta-train RMSE (optimistic): $21,627
v8 Kaggle RMSE for comparison:      $21,804.67


---
## 12. Generate Submission v9

In [ ]:
final_log  = meta_model.predict(meta_X_test)
final_pred = np.expm1(final_log)

sample_sub = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_v14 = pd.DataFrame({'Id': test['id'], 'Predicted': final_pred})
sub_v14 = sub_v14.set_index('Id').reindex(sample_sub['Id']).reset_index()

out = '../../outputs/submissions/sub_v14_features.csv'
sub_v14.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {sub_v14.shape}')
print(f'Price range: ${sub_v14.Predicted.min():,.0f} – ${sub_v14.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_v14.Predicted.mean():,.0f}')
print(f'\nv9 comparison: OOF $21,627 / Kaggle $21,755 — check if this improved.')

---
## 13. Summary

### Changes vs v8
| Change | Why | Expected Gain |
|---|---|---|
| `postal_sector_median_price` OOF | Replaced 9,125-level ordinal noise with a real geographic price signal | ~200–400 |
| `flat_model_median_price` OOF | Type S2 vs Improved price spread is ~$550K — direct signal | ~50–150 |
| `block_num` numeric | Replaced 2,514-level ordinal noise | ~50–100 |
| RandomizedSearchCV wider grid (40 trials) for XGB+LGBM | More combinations searched than v8's 30-trial grid | ~50–150 |

### Score tracker
| Version | Model | OOF RMSE | Kaggle |
|---|---|---|---|
| v6 | Log blend + OOF encoding | $21,818 | 22,124 |
| v7 | Stack XGB+LGBM | $21,841 | 21,906 |
| v8 | Stack XGB+LGBM+ET | _(see above)_ | 21,804.67 |
| **v9** | **Wider grid + richer encoding** | **_(run above)_** | **_(submit)_** |

### Key questions after running
- Did `postal_sector_median_price` reduce the OOF RMSE by >100? → confirms geographic signal value
- Are all Ridge coefficients positive and >0.05? → confirms 3-model diversity preserved
- Did the wider search grid find materially different params vs v8?

### Next steps if v9 improves
- [ ] Add CatBoost as 4th base model (native categorical handling, different architecture)
- [ ] Try `planning_area` OOF encoding (finer than `town`, coarser than `postal_sector`)
- [ ] Explore Optuna (Bayesian HPO) for even smarter hyperparameter search